<a href="https://colab.research.google.com/github/gayakarapetyan/PythonCourseH2/blob/main/Session3/00_the_Libraries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Libraries

**Environmental Statistics and Artificial Intelligence — Practical Python**


| Library | Job | One-line description |
|---|---|---|
| `numpy` | Fast math on arrays | The engine under everything |
| `pandas` | Tables (DataFrames) | Your Excel replacement, but scriptable |
| `matplotlib` | Plotting | Seeing your data |
| `scipy.stats` | Classical statistics | Tests, distributions, correlations |
| `statsmodels` | Statistical **modeling** | Regression with p-values, confidence intervals — the *statistician's* view |
| `scikit-learn` | Machine **learning** | Regression, classification, clustering, PCA — the *predictor's* view |
| `tensorflow / keras` | Neural networks |  |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# reproducibility: same "random" numbers for everyone in the room
np.random.seed(42)

print("numpy ", np.__version__)
print("pandas", pd.__version__)

---
## NumPy — arrays and fast math

NumPy gives us the `ndarray`: like a Python list, but it does math on **all elements at
once** (vectorized), and it is *much* faster. Every other library today is built on it.

In [ ]:
# 365 days of synthetic precipitation [mm/day]
precip = np.random.gamma(shape=0.6, scale=4.0, size=365)

print("type:", type(precip))
print("shape:", precip.shape)
print("first 5 days:", precip[:5].round(2))

In [ ]:
# Whole-array math in one line — no loops needed!
print("annual sum:", precip.sum().round(1), "mm")
print("daily mean:", precip.mean().round(2), "mm")
print("daily std:", precip.std().round(2), "mm")
print("max day:", precip.max().round(1), "mm")
print("95th pctile:", np.percentile(precip, 95).round(1), "mm  <- 'heavy rain' threshold")

### Boolean masking

A comparison like `precip > 10` returns an array of `True/False`. Use it as a *mask*
to filter — this is how you'll select flood days, dry spells, summer months… everywhere.

In [ ]:
heavy = precip > 10  # boolean array
print("days with > 10 mm:", heavy.sum())  # True counts as 1
print("their values:", precip[heavy][:5].round(1))
print("share of annual rain from these days:", round(100 * precip[heavy].sum() / precip.sum(), 1), "%")

### Task 1.1
A day is called **dry** if precipitation is below 1 mm.
1. What *fraction* of the year is dry? (Hint: the mean of a boolean array is a fraction!)
2. What is the mean precipitation **on wet days only**?

In [ ]:
# Your code here


---
## pandas — tables with names

NumPy arrays are anonymous numbers. **pandas DataFrames** add labels: column names, an
index, mixed data types. If your data looks like a spreadsheet, it belongs in a DataFrame.

In [ ]:
n = 100
elevation = np.random.uniform(20, 1200, n)                      # m a.s.l.
forest = np.random.uniform(5, 90, n)                         # % cover
area = np.random.lognormal(mean=5, sigma=1, size=n)        # km²
precip_mm = 500 + 0.6 * elevation + np.random.normal(0, 80, n)  # mm/yr (more rain uphill)
discharge = 0.4 * precip_mm - 1.5 * forest + np.random.normal(0, 60, n)

df = pd.DataFrame({
    "elevation_m": elevation.round(0),
    "forest_pct":  forest.round(1),
    "area_km2":    area.round(1),
    "precip_mm":   precip_mm.round(0),
    "discharge":   discharge.round(1),
})
df.head()

In [ ]:
# The first three commands for ANY new dataset:
print(df.shape)        # rows, columns
df.info()              # column types, missing values?

In [ ]:
df.describe().round(1)   # instant statistical summary of every numeric column

### Selecting, filtering, sorting

In [ ]:
# one column (a 'Series'):
df["discharge"].mean()

In [ ]:
# filtering = boolean masking, exactly like NumPy:
highland = df[df["elevation_m"] > 800]
print("highland catchments:", len(highland))
highland.sort_values("discharge").head(3)

In [ ]:
# pandas cut() function
x = pd.Series([5,20,30,56,78,96,120])
groups = pd.cut(x, bins = [0,50,100,125], labels = ['low','middle','high'])
print(groups)

In [ ]:
# groupby: split → compute → combine. THE pandas power tool.
df["zone"] = pd.cut(df["elevation_m"], bins=[0, 400, 800, 1300],
                    labels=["lowland", "midland", "highland"])
df.groupby("zone", observed=True)[["precip_mm", "discharge"]].mean().round(0)

### Task 2.1
1. Select all catchments with `forest_pct > 50`. How many are there?
2. Compute their mean `discharge` and compare it with the mean discharge of catchments
   with `forest_pct <= 50`. Which is higher? Does that match how we generated the data?

In [ ]:
# Your code here


---
## matplotlib — plotting the data

You already know the basics — quick refresher of the two plots you'll use constantly today

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(df["precip_mm"], df["discharge"], alpha=0.6, edgecolor="k")
axes[0].set_xlabel("Annual precipitation [mm]")
axes[0].set_ylabel("Discharge [mm/yr]")
axes[0].set_title("Scatter: is there a relationship?")

axes[1].hist(df["discharge"], bins=20, color="steelblue", edgecolor="k")
axes[1].set_xlabel("Discharge [mm/yr]")
axes[1].set_ylabel("Count")
axes[1].set_title("Histogram: how is it distributed?")

plt.show()

### Task 3.1
Make a scatter plot of `forest_pct` (x) vs `discharge` (y).


In [ ]:
# Your code here


---
## scipy.stats — classical statistics in one line

`scipy.stats` is the toolbox of distributions, tests and correlations.

In [ ]:
# Pearson correlation
r, p = stats.pearsonr(df["precip_mm"], df["discharge"])
print(f"Pearson r = {r:.2f}   (p-value = {p:.2g})")

# Spearman correlation
rho, p2 = stats.spearmanr(df["forest_pct"], df["discharge"])
print(f"Spearman rho = {rho:.2f} (p-value = {p2:.2g})")

> The p-value answers: *if there were truly no relationship, how likely is a correlation
> this strong by pure chance?* Small p (< 0.05 by convention) → the relationship is
> probably real. We'll go much deeper into dependency measures in the Regression notebook.

---
## statsmodels

`statsmodels` fits statistical models and tells you **everything** about them:
coefficients, uncertainties, p-values, R².

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols("discharge ~ precip_mm", data=df).fit() #ols(ordinary lest squares, it fits linear regression. it fits best fitting streight line by least squares)
print(model.summary())

That's a lot of numbers! The ones to read (we'll dissect this fully later):

- **coef** of `precip_mm` ≈ 0.4 → each extra mm of rain adds ~0.4 mm of discharge.
- **P>|t|** → is each coefficient significantly different from zero?
- **R-squared** → fraction of discharge variability explained by the model.

### Task 5.1
Extend the formula to `"discharge ~ precip_mm + forest_pct"` and refit.
1. What coefficient does `forest_pct` get?
2. Did R² improve compared to the precipitation-only model?

In [ ]:
# Your code here


---
## scikit-learn — machine learning with ONE interface

[scikit-learn](https://scikit-learn.org/stable/supervised_learning.html) is the standard ML
library in Python. It is **uniform API** — *every* model, from linear regression to random forests to clustering, works the same way:

```python
model = SomeModel(options)      # 1. choose & configure
model.fit(X, y)                 # 2. learn from data
model.predict(X_new)            # 3. predict new cases
model.score(X_test, y_test)     # 4. evaluate
```

- `X` = feature matrix (2-D: rows = samples, columns = features)
- `y` = target (1-D: one value per sample)

`X` is called input, what you give the model to make predictions from.
`y` is the output, its what you are trying to predict

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[["precip_mm", "forest_pct"]]
y = df["discharge"]

lr = LinearRegression().fit(X, y)

print("coefficients:", lr.coef_.round(3))
print("intercept:", round(lr.intercept_, 2))
print("R²:", round(lr.score(X, y), 3))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)   # 70% learn, 30% exam

lr = LinearRegression().fit(X_train, y_train)
print("R² on TRAIN:", round(lr.score(X_train, y_train), 3))
print("R² on TEST :", round(lr.score(X_test,  y_test),  3))

### Swap the model

**k-nearest-neighbors regression** (predict a catchment's discharge as the average of its
*k* most similar catchments). Only the first line changes:

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)
print("KNN  R² on TEST:", round(knn.score(X_test, y_test), 3))
print("Linear R² test :", round(lr.score(X_test, y_test), 3))

### Task 6.1
Try `n_neighbors` = 1, 5, 15, 50. For each, print R² on **train** and on **test**.
- What happens to the *train* score when `n_neighbors=1`? Why?
- Which value gives the best *test* score?
- This is your first encounter with **overfitting** — we'll formalize it in the next notebook!

In [ ]:
# Your code here
for k in [1, 5, 15, 50]:
    ...

### Preview: what ELSE can scikit-learn do?

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# synthetic 2-D data with 3 hidden groups
Xb, _ = make_blobs(n_samples=200, centers=3, random_state=7)

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xb)

plt.figure(figsize=(5.5, 4.5))
plt.scatter(Xb[:, 0], Xb[:, 1], c=km.labels_, cmap="viridis", alpha=0.7)
plt.scatter(*km.cluster_centers_.T, c="red", marker="X", s=200, label="centers")
plt.title("KMeans found the groups by itself — no labels given!")
plt.legend(); plt.show()

In [ ]:
# classification (predicting categories)
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

Xc, yc = make_classification(n_samples=300, n_features=5, n_informative=3,
                             random_state=1)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, random_state=1)

rf = RandomForestClassifier(random_state=0).fit(Xc_tr, yc_tr)   # .fit() again!
print("classification accuracy on test:", round(rf.score(Xc_te, yc_te), 3))

In [ ]:
# dimension reduction (PCA)
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(Xc)
print("variance explained by 2 components:",
      pca.explained_variance_ratio_.round(2), "→ together:",
      round(pca.explained_variance_ratio_.sum() * 100), "%")

---
## Cheat sheet

| You want to… | Library | Function / class |
|---|---|---|
| Fast array math, random data | `numpy` | `np.array`, `np.random.*`, `.mean()`, masking |
| Load / filter / summarize tables | `pandas` | `pd.read_csv`, `df.describe()`, `df.groupby()` |
| Scatter / histogram | `matplotlib` | `plt.scatter`, `plt.hist` |
| Correlations & tests | `scipy.stats` | `pearsonr`, `spearmanr` |
| Regression **with inference** | `statsmodels` | `smf.ols("y ~ x", df).fit().summary()` |
| Regression **for prediction** | `sklearn` | `LinearRegression`, `KNeighborsRegressor` |
| Honest evaluation | `sklearn` | `train_test_split`, `.score()` |
| Find groups (no labels) | `sklearn` | `KMeans` |
| Predict categories | `sklearn` | `RandomForestClassifier`, … |
| Compress dimensions | `sklearn` | `PCA` |
| Neural networks | `tensorflow.keras` | this afternoon!  |

---
---
## 🔑 Solutions (no peeking before trying!)

<details><summary>Click to expand</summary>

**Task 1.1**
```python
dry = precip < 1
print("dry fraction:", dry.mean())            # ~50-60%
print("mean on wet days:", precip[~dry].mean())
```

**Task 2.1**
```python
high_f = df[df["forest_pct"] > 50]
low_f  = df[df["forest_pct"] <= 50]
print(len(high_f))
print(high_f["discharge"].mean(), "vs", low_f["discharge"].mean())
# forested catchments have LOWER discharge — as generated (coefficient -1.5)
```

**Task 3.1**
```python
plt.scatter(df["forest_pct"], df["discharge"], alpha=0.6)
plt.xlabel("Forest [%]"); plt.ylabel("Discharge [mm/yr]")
# The trend is there but weaker-looking: the forest effect (-1.5 * forest, range ~130 mm)
# is smaller relative to the noise + precip variability than the precipitation effect.
```

**Task 5.1**
```python
m2 = smf.ols("discharge ~ precip_mm + forest_pct", data=df).fit()
print(m2.params)        # forest_pct close to -1.5 ✔
print(m2.rsquared)      # higher than before ✔
```

**Task 6.1**
```python
for k in [1, 5, 15, 50]:
    knn = KNeighborsRegressor(n_neighbors=k).fit(X_train, y_train)
    print(k, round(knn.score(X_train, y_train), 3),
             round(knn.score(X_test,  y_test),  3))
# k=1: train R² = 1.0 (memorized!) but poor test -> overfitting.
# Moderate k (≈5-15) usually best on test.
```
</details>